# Cell Maps for AI (CM4AI) Protein Compartment Localization Processing Notebook

The protein compartment localization data was downloaded from the [U2OS cell map portal](https://musicmaps.ai/u2os-cellmap-data/) on 9/24/25. Terms with more than 1000 proteins were discarded, and the rest were processed to use the annotated label if available and a GPT-4o generated label as a fallback.

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import re
import requests
import scipy.spatial.distance as dist
import seaborn as sns
import sys
from tqdm import tqdm
import scanpy as sc

from sklearn.feature_extraction.text import TfidfVectorizer
import anndata
from collections import OrderedDict

#OpenAI
import dotenv
from openai import OpenAI
import os

# Bokeh
from bokeh.io import output_notebook
from bokeh.plotting import figure, show, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.palettes import Category20
output_notebook()


from IPython.display import display, HTML, Markdown
dotenv.load_dotenv()
sys.setrecursionlimit(100000)

## Load and Pre-Process Data

In [ ]:
cm4ai = pd.read_csv('U2OS Cell Map Assemblies - Sheet1.tsv', sep='\t')
cm4ai

### Use Provided Names

In [ ]:
cm4ai.loc[:,'Annotation'] = cm4ai.loc[:,'Annotation'].fillna(cm4ai['GPT4 default name'])

counter = 1

for i,label in enumerate(cm4ai['Annotation']):
    if label=='System of unrelated proteins':
        cm4ai.iloc[i,1] = f'No annotation - {counter}'
        counter += 1

for i,row in cm4ai[cm4ai['Annotation'].duplicated('last')].iterrows():
    cm4ai.loc[i, 'Annotation'] += ' - 2'

cm4ai = cm4ai[(cm4ai['Number of Proteins']<1000)]
cm4ai

In [ ]:
cm4ai['Proteins'].map(str.split).map(len).mean()

In [ ]:
gene_info = pd.read_csv('../../tables/All_Mammalia.gene_info.gz', sep='\t', compression='gzip')
gene_info = gene_info[(gene_info['#tax_id']==9606) & (gene_info['type_of_gene']=='protein-coding')]
gene_info['Symbol'] = gene_info['Symbol'].str.upper()
symbols = set(gene_info['Symbol'])
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
synonyms = gene_info.copy()
synonyms['Synonyms'] = synonyms.apply(lambda x: x['Synonyms'].upper().split('|') + [x['Symbol']], axis=1)
synonyms = synonyms.explode('Synonyms')
synonyms['Synonyms'] = synonyms['Synonyms'].str.upper()
synonyms = synonyms[synonyms['Synonyms']!='-']
synonyms = synonyms.set_index('Synonyms')['Symbol'].to_dict()

In [ ]:
cm4ai_associations = cm4ai.copy()
cm4ai_associations['Proteins'] = cm4ai_associations['Proteins'].map(str.split)
cm4ai_associations = cm4ai_associations.explode('Proteins')
cm4ai_associations['Proteins'] = cm4ai_associations['Proteins'].map(str.upper).map(synonyms)
cm4ai_associations['Gene ID'] = cm4ai_associations['Proteins'].map(geneids)
cm4ai_associations = cm4ai_associations[['Proteins', 'Gene ID', 'Annotation', 'Assembly ID']].drop_duplicates().dropna().reset_index(drop=True)
cm4ai_associations.columns = ['Gene', 'Gene ID', 'Assembly', 'Assembly ID']
cm4ai_associations

In [ ]:
cm4ai_associations.nunique()

## Process Data for SQL

### Resource

In [ ]:
(122, 'Cell Maps for AI', 'CM4AI', 'The CM4AI data generation project seeks to map the spatiotemporal architecture of human cells and use these maps toward the grand challenge of interpretable genotype-phenotype learning', 'Mapping Cellular Structure and Function', 'https://cm4ai.org/')

### Dataset

In [ ]:
#(id, name, name_without_resource, description, association, gene_set_description, gene_sets_description, attribute_set_description, is_signed, is_continuous_valued, last_updated, directory, num_page_views, resource_fk, measurement_fk, dataset_group_fk, attribute_type_fk, attribute_group_fk, evidence_type, evidence_group, measurement_bias, attribute_type_plural)
(191, 'CM4AI U2OS Cell Map Protein Localization Assemblies', 'U2OS Cell Map Protein Localization Assemblies', 'Protein localization assemblies constructed from integrating AP-MS biomolecular interaction and IF imaging data', 'protein-biological term associations from integrated AP-MS interactions and IF imaging', 'proteins comprising {0} assembly using integrated AP-MS and IF data from the CM4AI U2OS Cell Map Protein Localization Assemblies dataset.', 'sets of proteins comprising assemblies from integrated AP-MS and IF data from the CM4AI U2OS Cell Map Protein Localization Assemblies dataset.', 'assemblies containing {0} protein from integrated AP-MS and IF data from the CM4AI U2OS Cell Map Protein Localization Assemblies dataset.', 0, 0, '2025-09-29', 'cm4aiu2os', 0, 122, 3, 6, 18, 4, 'association by data aggregation', 'primary experimental data', 'high throughput, data driven', 'biological terms')

### Publication

In [ ]:
#(id, long_citation, short_citation, url, pmid, pubmed_url, first_author_last_name, first_author_initials, journal_abbreviation, year, title, volume, pages)
(173, 'Schaffer LV, et al. (2025) Multimodal cell maps as a foundation for structural and functional genomics. Nature. 642:222–231', 'Schaffer, Nature, 2025', 'dx.doi.org/10.1038/s41586-025-08878-3', 40205054, 'https://ncbi.nlm.nih.gov/pubmed/40205054', 'Schaffer', 'LV', 'Nature', 2025, 'Multimodal cell maps as a foundation for structural and functional genomics', 642, '222-231')

(279, 191, 173)

### Genes

In [ ]:
index = 58933
genes = pd.read_csv('../../tables/gene.txt',sep='\t')
genes['symbol'] = genes['symbol'].str.upper()
genelist = set(genes['symbol'])
genefks = genes.set_index('symbol')['id'].to_dict()
gene_info_dict = gene_info.set_index('Symbol')[['GeneID', 'description']].to_dict()

for gene in cm4ai_associations['Gene'].unique():
    if gene not in genelist:
        print((index, gene, gene_info_dict['GeneID'][gene], gene_info_dict['description'][gene], f"https://ncbi.nlm.nih.gov/gene/{gene_info_dict['GeneID'][gene]}"), end=',\n')
        genefks[gene] = index
        index += 1

### Attributes

In [ ]:
index = 558934
attributes = pd.read_csv('../../tables/attribute.txt', sep='\t')
attributes['name_from_naming_authority'] = attributes['name_from_naming_authority'].astype(str).map(str.upper)
attribute_fks = attributes.set_index('name_from_naming_authority')['id'].to_dict()
assemblyids = cm4ai.set_index('Annotation')['Assembly ID'].to_dict()

for assembly in cm4ai_associations['Assembly'].unique():
    if assembly.upper() not in attribute_fks:
        print((index, assembly, assemblyids[assembly], 117), end=',\n')
        attribute_fks[assembly.upper()] = index
        index += 1

### Gene Set

In [ ]:
index = 139200000
gene_set_fks = {}

for assembly in cm4ai_associations['Assembly'].unique():
        print((index, assembly, assemblyids[assembly], 191, 18, attribute_fks[assembly.upper()]), end=',\n')
        gene_set_fks[assembly] = index
        index += 1

### Associations

In [ ]:
index = 96000000

associations = cm4ai_associations.copy()
associations['Gene'] = associations['Gene'].apply(lambda x: genefks[x])
associations['Assembly'] = associations['Assembly'].apply(lambda x: gene_set_fks[x])
associations['Threshold'] = 1
associations = associations.get(['Gene', 'Assembly', 'Threshold'])
associations.columns = ['gene_fk', 'gene_set_fk', 'threshold_value']
associations = associations.drop_duplicates().reset_index(drop=True)
associations.index += index
associations = associations.rename_axis('id')
associations.to_csv('../../harmonizome-update/cm4aiu2os.csv')
associations

In [ ]:
associations.nunique()

## Create Downloads

In [ ]:
output_path = 'downloads'

### Binary Matrix

In [ ]:
binarymatrix = pd.crosstab(cm4ai_associations['Gene'], cm4ai_associations['Assembly']).replace(np.nan, 0)
binarymatrixT = binarymatrix.T
binarymatrix.to_csv(f'{output_path}/gene_attribute_matrix.txt.gz', sep='\t', compression='gzip')
binarymatrix

### Gene Attribute Edge List

In [ ]:
edgelist = cm4ai_associations.copy()
edgelist['Threshold Value'] = 1
edgelist.to_csv(f'{output_path}/gene_attribute_edges.txt.gz', sep='\t', compression='gzip')
edgelist

### Gene List

In [ ]:
geneslist = edgelist[['Gene', 'Gene ID']].drop_duplicates().reset_index(drop=True)
geneslist.to_csv(f'{output_path}/gene_list_terms.txt.gz', sep='\t', compression='gzip')
geneslist

### Attribute List

In [ ]:
attributeslist = edgelist[['Assembly', 'Assembly ID']].drop_duplicates().reset_index(drop=True)
attributeslist.to_csv(f'{output_path}/attribute_list_entries.txt.gz', sep='\t', compression='gzip')
attributeslist

### Gene Set Library

In [ ]:
with open(f'{output_path}/gene_set_library_crisp.gmt', 'w') as f:
    arr = binarymatrix.reset_index(drop=True).to_numpy(dtype=np.int_)
    attributes = binarymatrix.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrix.index[arr[:, i] == 1]])>= 5:
            print(attributes[i], '', *binarymatrix.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Attribute Set Library

In [ ]:
with open(f'{output_path}/attribute_set_library_crisp.gmt', 'w') as f:
    arr = binarymatrixT.reset_index(drop=True).to_numpy(dtype=np.int_)
    genes = binarymatrixT.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrixT.index[arr[:, i] == 1]])>= 5:
            print(genes[i], '', *binarymatrixT.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Gene Similarity Matrix

In [ ]:
gene_similarity_matrix = dist.pdist(binarymatrix.to_numpy(dtype=np.int_), 'cosine')
gene_similarity_matrix = dist.squareform(gene_similarity_matrix)
gene_similarity_matrix = 1 - gene_similarity_matrix

gene_similarity_matrix = pd.DataFrame(data=gene_similarity_matrix, index=binarymatrix.index, columns=binarymatrix.index)
gene_similarity_matrix.index.name = None
gene_similarity_matrix.columns.name = None
gene_similarity_matrix.to_csv(f'{output_path}/gene_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
gene_similarity_matrix

### Attribute Similarity Matrix

In [ ]:
attribute_similarity_matrix = dist.pdist(binarymatrixT.to_numpy(dtype=np.int_), 'cosine')
attribute_similarity_matrix = dist.squareform(attribute_similarity_matrix)
attribute_similarity_matrix = 1 - attribute_similarity_matrix

attribute_similarity_matrix = pd.DataFrame(data=attribute_similarity_matrix, index=binarymatrixT.index, columns=binarymatrixT.index)
attribute_similarity_matrix.index.name = None
attribute_similarity_matrix.columns.name = None
attribute_similarity_matrix.to_csv(f'{output_path}/attribute_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
attribute_similarity_matrix

### Knowledge Graph Serialization

In [ ]:
nodes = {}
edges = []

for gene in geneslist.index:
    gene = geneslist.loc[gene]
    nodes[int(gene['Gene ID'])] = {
        "type":"gene",
        "properties": {
            "id":int(gene['Gene ID']),
            "label":gene['Gene']
        }}

for pert in attributeslist.index:
    pert = attributeslist.loc[pert]
    nodes[pert['Assembly ID']] = {
        "type":"biological term",
        "properties": {
            "label":pert['Assembly'],
            "id":pert['Assembly ID']
        }}

for edge in edgelist.index:
    edge = edgelist.loc[edge]
    edges.append({
        "source": str(edge['Assembly ID']),
        "relation": "has component",
        "target": int(edge['Gene ID']),
        "properties":{
            "id": str(edge['Assembly ID'])+":"+str(edge['Gene ID']),
            "source_label":edge['Assembly'],
            "source_id": str(edge['Assembly ID']),
            "target_id": int(edge['Gene ID']),
            "target_label":edge['Gene'],
            "directed":True,
            "threshold":1,
        }})

#### RDF

In [ ]:
with open(output_path+'/kg_serializations/cm4aiu2os.rdf', 'w') as f:
    print('@prefix gene: <https://ncbi.nlm.nih.gov/gene/> .', file=f)
    print('@prefix RO: <http://purl.obolibrary.org/RO_> .', file=f)
    print('@prefix CM4AI: <https://www.ndexbio.org/viewer/networks/f693137a-d2d7-11ef-8e41-005056ae3c32?queryType=firstStepNeighborhood&maximizeResultView=true&query=> .', file=f)
    
    print('', file=f)
    for edge in edges:
        source_id = edge['properties']['source_id']
        print(f"CM4AI:{source_id}",'RO:0002180', f"gene:{edge['properties']['target_label']}", end=' .\n', file=f)

#### JSON

In [ ]:
with open(output_path+'/kg_serializations/cm4aiu2os.json', 'w') as f:
    serial = json.dump(
        {
            "Version":"1", 
            "nodes": nodes,
            "edges": edges
        }, indent=4, fp=f)

#### TSV

In [ ]:
nodeframe = pd.DataFrame(nodes).T
nodeframe['id'] = nodeframe['properties'].apply(lambda x: x['id'])
nodeframe['label'] = nodeframe['properties'].apply(lambda x: x['label'])
namespace = {'gene':'NCBI Entrez', 'biological term':'CM4AI'}
nodeframe['namespace'] = nodeframe['id'].map(namespace)
nodeframe = nodeframe.get(['namespace', 'id', 'label']).reset_index(drop=True)
nodeframe.to_csv(output_path+'/kg_serializations/cm4aiu2os_tsv/nodes.tsv', sep='\t')
nodeframe

In [ ]:
edgeframe = pd.DataFrame(edges)
edgeframe['threshold'] = edgeframe['properties'].apply(lambda x: x['threshold'])
edgeframe = edgeframe.get(['source', 'relation', 'target','threshold'])
edgeframe.to_csv(output_path+'/kg_serializations/cm4aiu2os_tsv/edges.tsv', sep='\t')
edgeframe

## Prepare Visualizations

### Gene-Attribute Clustermap

In [ ]:
sns.clustermap(binarymatrix, cmap='seismic', center=0, figsize=(10,10), xticklabels=False, yticklabels=False)

### Gene-Gene Similarity Clustermap

In [ ]:
ax = sns.clustermap(gene_similarity_matrix, cmap='seismic', center=0)

### Attribute-Attribute Similarity Clustermap

In [ ]:
sns.clustermap(attribute_similarity_matrix, cmap='seismic', center=0, xticklabels=False, yticklabels=False)

### UMAP

In [ ]:
def load_gmt(file):
    gmt = OrderedDict()
    for line in file:
        term, blank, *geneset = line.strip().split('\t')
        gmt[term] = ' '.join(set(geneset))
    return gmt
libdict = load_gmt(open(f'{output_path}/gene_set_library_crisp.gmt', 'r'))
scatterdir = 'images/'

In [ ]:
def process_scatterplot(libdict, nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1):
    print("\tTF-IDF vectorizing gene set data...")
    vec = TfidfVectorizer(max_df=maxdf, min_df=mindf)
    X = vec.fit_transform(libdict.values())
    print(X.shape)
    adata = anndata.AnnData(X)
    adata.obs.index = libdict.keys()

    print("\tPerforming Leiden clustering...")
    ### the n_neighbors and min_dist parameters can be altered
    sc.pp.neighbors(adata, n_neighbors=nneighbors, use_rep='X')
    sc.tl.leiden(adata, resolution=1.0)
    sc.tl.umap(adata, min_dist=mindist, spread=spread, random_state=42)

    new_order = adata.obs.sort_values(by='leiden').index.tolist()
    adata = adata[new_order, :]
    adata.obs['leiden'] = 'Cluster ' + adata.obs['leiden'].astype('object')

    df = pd.DataFrame(adata.obsm['X_umap'])
    df.columns = ['x', 'y']

    df['cluster'] = adata.obs['leiden'].values
    df['term'] = adata.obs.index
    df['genes'] = [libdict[l] for l in df['term']]

    return df

In [ ]:
def get_scatter_colors(df):
    clusters = pd.unique(df['cluster']).tolist()
    colors = list(Category20[20])[::2] + list(Category20[20])[1::2]
    color_mapper = {clusters[i]: colors[i % 20] for i in range(len(clusters))}
    return color_mapper

def get_scatterplot(scatterdf):
    df = scatterdf.copy()
    color_mapper = get_scatter_colors(df)
    df['color'] = df['cluster'].apply(lambda x: color_mapper[x])

    hover_emb = HoverTool(name="df", tooltips="""
        <div style="margin: 10">
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Gene Set:</span>
                <span style="font-size: 12px">@gene_set</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Coordinates:</span>
                <span style="font-size: 12px">(@x,@y)</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Cluster:</span>
                <span style="font-size: 12px">@cluster</span>
            </div>
        </div>
    """)
    tools_emb = [hover_emb, 'pan', 'wheel_zoom', 'reset', 'save']

    plot_emb = figure(
        width=1000, 
        height=700, 
        tools=tools_emb
    )

    source = ColumnDataSource(
        data=dict(
            x = df['x'],
            y = df['y'],
            gene_set = df['term'],
            cluster = df['cluster'],
            colors = df['color'],
            label = df['cluster']
        )
    )

    # hide axis labels and grid lines
    plot_emb.xaxis.major_tick_line_color = None
    plot_emb.xaxis.minor_tick_line_color = None
    plot_emb.yaxis.major_tick_line_color = None
    plot_emb.yaxis.minor_tick_line_color = None
    plot_emb.xaxis.major_label_text_font_size = '0pt'
    plot_emb.yaxis.major_label_text_font_size = '0pt' 

    plot_emb.output_backend = "svg"    
    
    plot_emb.title = 'Gene Sets in the CM4AI U2OS Cell Map Protein Localization Assemblies Library'
    plot_emb.xaxis.axis_label = "UMAP_1"
    plot_emb.yaxis.axis_label = "UMAP_2"
    plot_emb.xaxis.axis_label_text_font_style = 'normal'
    plot_emb.xaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_style = 'normal'
    plot_emb.title.align = 'center'
    plot_emb.title.text_font_size = '18px'
    
    s = plot_emb.scatter(
        'x', 
        'y', 
        size = 4, 
        source = source, 
        color = 'colors'
    )
    
    return plot_emb

In [ ]:
## defaults: nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1
scatter_df = process_scatterplot(libdict, nneighbors=5,mindist=0.1
     ,spread=0.5
     ,maxdf=0.5
     ,mindf=1
)

# Display Scatter Plot
plot = get_scatterplot(scatter_df)
output_notebook()
show(plot)

In [ ]:
output_file(filename=f"{scatterdir}/umap.html", title = 'Gene Sets in CM4AI U2OS Cell Map Protein Localization Assemblies Library')
save(plot)